In [ ]:
import pandas as pd
import numpy as np
import os
import itertools as it
from snp_analysis_tools_sherlock import *
from coalescence_analysis_tools import *
import iqplot
import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from bokeh.layouts import gridplot

hv.extension('bokeh')

In [ ]:
fname = '~/git/coalescence-pilot-mgx/workflow/out/midas2_output/mergev4/species/species_relative_abundance.tsv'
df_abundance = pd.read_csv(fname, delimiter = '\t', low_memory=False)
df_abundance
fname = '~/git/coalescence-pilot-mgx/workflow/out/midas2_output/old_species/species/metadata.tsv'
df_metadata= pd.read_csv(fname, delimiter = '\t')
df_metadata

def transform_df(df_abundance):
    df_abundance['Lineage'] = df_abundance['species_id'].transform(lambda x: df_metadata.loc[df_metadata['species_id'] == x,'Lineage'].values[0])
    df_abundance['species'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-1])
    df_abundance['genus'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-2])
    df_abundance['family'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-3])
    df_abundance['phyla'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[1])
    return df_abundance

df_metadata = transform_df(df_metadata)
#df_abundance_melted = pd.melt(df_abundance, 
 #                                 var_name = 'sample', 
  #                                id_vars='species_id',
   #                               value_name = 'relative_abundance')


#df_abundance_melted = transform_df(df_abundance_melted)
df_abundance_melted= pd.read_csv('all_abundances_melted.csv', )

In [ ]:
df_metadata.loc[df_metadata['family']=='f__CAG-74',:]

In [ ]:
df_abundance_melted['sample'].unique()

In [ ]:
BF['counts']=1
BFg=BF.loc[BF['relative_abundance']>0.,:]
BFg.groupby(['family']).sum(numeric_only=True).sort_values(by='relative_abundance',ascending=False,)
fams = BFg['family'].unique()
len(fams)

In [ ]:
BF=df_abundance_melted.loc[df_abundance_melted['sample']=='BF-Fecal',:]
cmap_family= {'f__Bacteroidaceae': '#8dd3c7',
 'f__Ruminococcaceae': '#ffffb3',
 'f__Acutalibacteraceae': '#bebada',
 'f__Peptoniphilaceae': '#fb8072',
 'f__Oscillospiraceae': '#80b1d3',
 'f__Enterococcaceae': '#fdb462',
 'f__Tannerellaceae': '#b3de69',
 'f__Lachnospiraceae': '#fccde5',
 'f__Veillonellaceae': '#bc80bd',
 'f__Peptostreptococcaceae': '#ccebc5',
 'f__Acidaminococcaceae': '#ffed6f',
 'f__other': '#d9d9d9'}
cmap_family = {}
for i,fam in enumerate(fams):
    cmap_family[fam]=bokeh.palettes.Set3[12][i]
    

In [ ]:
BF=df_abundance_melted.loc[df_abundance_melted['sample']=='BF-Fecal',:]
BF.to_csv('BF_sp_abundances_with_zeros.csv',index=False)

In [ ]:
BFg=BF.loc[BF['relative_abundance']>1e-4,:]
BFg.sort_values(by='relative_abundance',ascending=False)


In [ ]:

BF=df_abundance_melted.loc[df_abundance_melted['sample']=='BF-Fecal',:]
BFg=BF.loc[BF['relative_abundance']>1e-2,:]
BFg.groupby(['family']).sum(numeric_only=True).sort_values(by='relative_abundance',ascending=False,)
fams = BFg['family'].unique()
len(fams)
cmap_family = {}
fams = ['f__Acutalibacteraceae','f__Bacteroidaceae',
        'f__Bifidobacteriaceae','f__CAG-239','f__CAG-508','f__CAG-74',
        'f__Erysipelotrichaceae','f__Lachnospiraceae','f__Muribaculaceae',
        'f__Rikenellaceae','f__Ruminococcaceae']

fams = ['f__Bacteroidaceae', 'f__Lachnospiraceae','f__Ruminococcaceae',
        'f__Erysipelotrichaceae', 'f__Muribaculaceae', 'f__CAG-239','f__Bifidobacteriaceae',
        'f__CAG-74','f__Rikenellaceae','f__CAG-508','f__Acutalibacteraceae',]
for i,fam in enumerate(fams):
    cmap_family[fam]=bokeh.palettes.Set3[12][i]
cmap_family['f__other']= bokeh.palettes.Set1[9][-1]

In [ ]:
BFg['relative_abundance'].min()

In [ ]:
def ecdf_transform(data):
    return 1- data.rank(method="first") / len(data)

BFg.loc[:, "rel abun ECDF"] = BFg[
    "relative_abundance"
].transform(ecdf_transform)

p1=hv.Scatter(
    data=BFg,
    kdims='relative_abundance',
    vdims=[('rel abun ECDF', 'ECDF'), ],
).opts(width=200,height=200, color= 'grey',xlabel='Relative Abundance',
       logx=True,
      ylabel='1-P(f>x)')
p1


In [ ]:
p = iqplot.ecdf(BFg['relative_abundance'], complementary=True)

bokeh.io.show(p)

In [ ]:
def adjust_df(meso_df):
    for sample in meso_df['sample'].unique():
        full= meso_df.loc[meso_df['sample']==sample,'relative_abundance'].sum()

        meso_df.loc[meso_df['sample']==sample,'relative_abundance']=  meso_df.loc[meso_df['sample']==sample,'relative_abundance']/full
        print(full,meso_df.loc[meso_df['sample']==sample,'relative_abundance'].sum())
    return meso_df
good_families = list(cmap_family.keys())
good_families

#BF.loc[BF['relative_abundance']<1e-2,'relative_abundance']=0
#print(df_meso['family'])
BF = adjust_df(BF)
#print(df_meso['family'])
BF['family_plot']=BF['family'].copy()

BF.loc[~BF['family'].isin(good_families),'family_plot']='f__other'
BF=BF.sort_values(by='family_plot')

bars2 = hv.Bars(BF, kdims=['sample', 'species_id',],
               vdims = ['relative_abundance','family_plot',])

bars2=bars2.opts(width=400, height=400).opts(stacked=True,#alpha='relative_abundance',
                                      color='family_plot',
                                      cmap=cmap_family,
                                      alpha=1.,
                                             bar_width = 2,
                                      xlabel=None,
                                      ylabel='Relative Abundance',
                                        show_legend=True,legend_position='right',
                                         )#.sort(by='passage_plot')

bars2

In [ ]:
df_metadata

In [ ]:
fnames = glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/calculateDiversityDepthv3/*/*_diversity_df1.csv')
dfs = []

for fname in fnames:
    df = pd.read_csv(fname).rename(columns={'0':'div'})
    sp = int(fname.split('/')[-2])
    df['species_id']=sp
    df['sp_name']= df_metadata.loc[df_metadata['species_id']==sp,'species'].values[0]
    dfs.append(df)
all_df= pd.concat(dfs)
all_df_BF = all_df.loc[all_df['Unnamed: 0']=='BF-Fecal',:]
all_df_BF.columns.values

In [ ]:
all_df_BF.loc[:, "div ECDF"] = all_df_BF[
    'div'
].transform(ecdf_transform)

p1=hv.Scatter(
    data=all_df_BF,
    kdims='div',
    vdims=[('div ECDF', 'ECDF'), ],
).opts(width=200,height=200, color= 'black',xlabel='SNP Diversity',
       logx=True,
      ylabel='1-P(f>x)')
p1 = hv.render(p1)
p1.ray(x = 1e-3,y=-1,angle=np.pi/2,color='grey',)
bokeh.io.show(p1)


In [ ]:
len(all_df_BF.loc[all_df_BF['div']>1e-3,:])

In [ ]:
all_df_BF.sort_values(by='div',ascending=False)